<a href="https://colab.research.google.com/github/ennaguiyoussef/otto-recommender-system/blob/main/notebooks/run_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OTTO – Multi-Objective Recommender System (Colab Runner)

2-stage recommender pipeline (**Polars** + **3-Matrix Co-visitation Candidate Generation** + **LightGBM & CUDA GPU XGBoost Reranker Ensemble**).

**Recommended Colab Runtime:** `Runtime -> Change runtime type -> L4 GPU (or A100 / T4 GPU) + High-RAM ON`.

**Storage Architecture:** All datasets, intermediate Parquet artifacts, trained models, and submission files are persisted directly on Google Drive under `/content/drive/MyDrive/OTTO/`.

## 1. Mount Google Drive

In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


## 2. Clone or Update Repository

In [2]:
import os

# For a private repo, store a GitHub Personal Access Token in Colab Secrets as 'GITHUB_TOKEN'
try:
    from google.colab import userdata
    gh_token = userdata.get("GITHUB_TOKEN")
except Exception:
    gh_token = None

REPO_URL = (
    f"https://{gh_token}@github.com/ennaguiyoussef/otto-recommender-system.git"
    if gh_token
    else "https://github.com/ennaguiyoussef/otto-recommender-system.git"
)
REPO_DIR = "/content/otto-recommender-system"

if os.path.exists(os.path.join(REPO_DIR, ".git")):
    print(f"Repository already exists at {REPO_DIR}. Pulling latest changes...")
    %cd {REPO_DIR}
    !git pull
else:
    print(f"Cloning repository into {REPO_DIR}...")
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

Cloning repository into /content/otto-recommender-system...
Cloning into '/content/otto-recommender-system'...
remote: Enumerating objects: 103, done.
remote: Counting objects: 100% (103/103), done.
remote: Compressing objects: 100% (59/59), done.
remote: Total 103 (delta 42), reused 97 (delta 36), pack-reused 0 (from 0)
Receiving objects: 100% (103/103), 50.04 KiB | 4.55 MiB/s, done.
Resolving deltas: 100% (42/42), done.
/content/otto-recommender-system


## 3. Install Dependencies

In [3]:
!pip install -q -U kaggle
!pip install -q -r requirements.txt

## 4. Configure Kaggle API Token & Download Parquet Dataset to Google Drive

Reads your Kaggle API token (`KGAT_...`) from `/content/drive/MyDrive/OTTO/kaggle_token.txt`, configures `~/.kaggle/access_token` + `KAGGLE_API_TOKEN`, and downloads `radek1/otto-full-optimized-memory-footprint` directly into `/content/drive/MyDrive/OTTO/data/raw/` if not already present.

In [4]:
import glob
import os

BASE_DRIVE_DIR = "/content/drive/MyDrive/OTTO"
KAGGLE_TOKEN_SRC = os.path.join(BASE_DRIVE_DIR, "kaggle_token.txt")
RAW_DATA_DIR = os.path.join(BASE_DRIVE_DIR, "data", "raw")
DATASET_SLUG = "radek1/otto-full-optimized-memory-footprint"

os.makedirs(RAW_DATA_DIR, exist_ok=True)

# Configure Kaggle API Token (KGAT_...)
if not os.path.exists(KAGGLE_TOKEN_SRC):
    raise FileNotFoundError(
        f"Missing Kaggle API token file at {KAGGLE_TOKEN_SRC}."
    )

with open(KAGGLE_TOKEN_SRC, "r", encoding="utf-8") as f:
    kaggle_token = f.read().strip()

os.environ["KAGGLE_API_TOKEN"] = kaggle_token
kaggle_cfg_dir = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_cfg_dir, exist_ok=True)
access_token_dst = os.path.join(kaggle_cfg_dir, "access_token")
with open(access_token_dst, "w", encoding="utf-8") as f:
    f.write(kaggle_token)
os.chmod(access_token_dst, 0o600)
print("Kaggle API token configured successfully.")

# Download optimized Parquet dataset if not already present on Drive
existing_parquets = glob.glob(os.path.join(RAW_DATA_DIR, "**", "*.parquet"), recursive=True)
if existing_parquets:
    print(f"Found {len(existing_parquets)} Parquet file(s) in {RAW_DATA_DIR}. Skipping download.")
else:
    print(f"Downloading {DATASET_SLUG} into {RAW_DATA_DIR}...")
    !kaggle datasets download -d {DATASET_SLUG} -p "{RAW_DATA_DIR}" --unzip
    print("Download and extraction complete!")

Kaggle API token configured successfully.
Found 2 Parquet file(s) in /content/drive/MyDrive/OTTO/data/raw. Skipping download.


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 5. Run End-to-End Pipeline (`sample` Mode)

In [6]:
!python main.py --mode sample --stage candidates,features,train,predict

2026-11-25 18:42:27 | INFO     | otto.main | Starting OTTO pipeline | mode='sample' | stage='candidates,features,train,predict' | config='configs/config.yaml'
2026-11-25 18:42:27 | INFO     | src.utils.config_loader | Initialized storage directories under: /content/drive/MyDrive/OTTO
2026-11-25 18:42:27 | INFO     | otto.main | === Running stage: CANDIDATES ===
2026-11-25 18:44:19 | INFO     | src.candidates.covisitation | [candidates] Streaming 'train.parquet' (171266098 rows) into 16 disk buckets...
2026-11-25 18:44:56 | INFO     | src.candidates.covisitation | [candidates] Streaming 'history.parquet' (384620 rows) into 16 disk buckets...
2026-11-25 18:46:22 | INFO     | src.candidates.covisitation | [candidates] Processed session bucket 4/16
2026-11-25 18:47:46 | INFO     | src.candidates.covisitation | [candidates] Processed session bucket 8/16
2026-11-25 18:49:11 | INFO     | src.candidates.covisitation | [candidates] Processed session bucket 12/16
2026-11-25 18:50:34 | INFO     |

In [ ]:
!python main.py --mode sample --stage all

## 6. Run Full Pipeline (ull Mode — Auto-Resume Checkpoints Enabled)

All stages, buckets, feature shards, models, and prediction batches checkpoint atomically on Google Drive.
**If Colab disconnects or GPU stops mid-run:** simply reconnect (GPU or CPU) and re-run !python main.py --mode full --stage all — it automatically resumes from the exact unfinished step!

In [ ]:
# Uncomment when ready to train and generate submissions on 100% of the data:
# !python main.py --mode full --stage all